In [1]:
!pip -q install langchain langchain-google-genai python-dotenv pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.2 MB/s eta 0:00:00


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

In [3]:
from google.colab import userdata
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

#Model

In [4]:
llm=ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=GOOGLE_API_KEY,
    temperature=0.5
)


# Create Schema (PydanticModel)

In [7]:
from pydantic import BaseModel,Field

In [9]:
class Course(BaseModel):
  title:str =Field(
      description="Course Name"
  )
  duration:str =Field(
      description="Course Duration"
  )
  level: str =Field(
      description="Course Level"
  )
  teacher : str =Field(
      description="teacher name "
  )
  price : str=Field (
      description="Course Price"
  )

In [10]:
from langchain_core.output_parsers import PydanticOutputParser
parser=PydanticOutputParser(pydantic_object=Course)

In [12]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"title": {"description": "Course Name", "title": "Title", "type": "string"}, "duration": {"description": "Course Duration", "title": "Duration", "type": "string"}, "level": {"description": "Course Level", "title": "Level", "type": "string"}, "teacher": {"description": "teacher name ", "title": "Teacher", "type": "string"}, "price": {"description": "Course Price", "title": "Price", "type": "string"}}, "required": ["title", "duration", "level", "teacher", "price"]}\n```'

# Create prompt

In [13]:
from IPython.core.oinspect import Inspector
prompt=ChatPromptTemplate.from_template(
    """
    You are AI Course Generator
    Generate a professional Online Course
    {format_instructions}
    Topic {topic}
    """
).partial(
    format_instructions=parser.get_format_instructions()
)

In [14]:
chain=prompt | llm | parser

In [16]:
response=chain.invoke(
    {
        "topic":"Data Science"
    }
)
response

Course(title='Mastering Data Science: From Foundations to Advanced Machine Learning', duration='8 Weeks (Approx. 40 Hours of Content)', level='Intermediate', teacher='Dr. Anya Sharma, Lead Data Scientist', price='$299.99')

In [17]:
response=chain.invoke(
    {
        "topic":"ReinForcement Learning"
    }
)
response

Course(title='Reinforcement Learning: Foundations and Advanced Applications', duration='40 hours (8 weeks)', level='Intermediate', teacher='Dr. Alex Chen', price='$199')

# Complete LCEL flow


```
               User
                 │
                 ▼
         Topic = RL
                 │
                 ▼
      ChatPromptTemplate
                 │
                 ▼
       Partial Prompt
(format_instructions Added)
                 │
                 ▼
             Gemini
                 │
      JSON Response
                 │
                 ▼
      PydanticOutputParser
                 │
                 ▼
         Course Object
                 │
                 ▼
        Python Variables
```

